# Mölnlycke Health Care Demo - Semantic View & Cortex Agent

This notebook creates:
1. **Semantic View** for Cortex Analyst (natural language to SQL)
2. **Cortex Search Service** for clinician review search
3. **Cortex Agent** for Snowflake Intelligence

In [ ]:
import os
from snowflake.snowpark import Session

session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "oregon_tp")).create()
print(f"Connected as: {session.get_current_user()}")
print(f"Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")

In [ ]:
session.sql("USE DATABASE MOLNLYCKE_DEMO").collect()
session.sql("USE SCHEMA ANALYTICS").collect()
print("Using MOLNLYCKE_DEMO.ANALYTICS")

## Step 1: Create Semantic View for Cortex Analyst

In [ ]:
semantic_view_ddl = """
CREATE OR REPLACE SEMANTIC VIEW MOLNLYCKE_DEMO.ANALYTICS.MOLNLYCKE_COMPETITIVE_INTEL

TABLES (
    PRODUCTS AS MOLNLYCKE_DEMO.ANALYTICS.DIM_PRODUCTS 
        PRIMARY KEY (PRODUCT_ID)
        COMMENT = 'MedTech products - our 10 Mölnlycke products + 15 competitor products',
    
    REVIEWS AS MOLNLYCKE_DEMO.ANALYTICS.REVIEWS 
        PRIMARY KEY (REVIEW_ID)
        COMMENT = 'Clinician product reviews with ratings for wound care and surgical products',
    
    PRODUCT_METRICS AS MOLNLYCKE_DEMO.ANALYTICS.FACT_PRODUCT_METRICS 
        PRIMARY KEY (PRODUCT_ID, YEAR_QUARTER)
        COMMENT = 'Quarterly product performance metrics in European markets'
)

RELATIONSHIPS (
    REVIEWS(PRODUCT_ID) REFERENCES PRODUCTS(PRODUCT_ID),
    PRODUCT_METRICS(PRODUCT_ID) REFERENCES PRODUCTS(PRODUCT_ID)
)

FACTS (
    PRODUCTS.EU_HOSPITAL_COUNT AS EU_HOSPITAL_COUNT WITH SYNONYMS = ('hospitals', 'accounts', 'facilities') COMMENT = 'Number of EU hospitals using this product',
    REVIEWS.RATING AS RATING WITH SYNONYMS = ('stars', 'score') COMMENT = 'Star rating 1-5',
    PRODUCT_METRICS.EU_UNITS_SOLD AS EU_UNITS_SOLD WITH SYNONYMS = ('units sold', 'volume', 'units') COMMENT = 'Quarterly units sold in EU',
    PRODUCT_METRICS.EU_REVENUE_EUR AS EU_REVENUE_EUR WITH SYNONYMS = ('revenue', 'sales') COMMENT = 'Quarterly revenue in EUR',
    PRODUCT_METRICS.AVG_SELLING_PRICE_EUR AS AVG_SELLING_PRICE_EUR WITH SYNONYMS = ('avg price', 'average price', 'ASP') COMMENT = 'Average selling price in EUR',
    PRODUCT_METRICS.EU_MARKET_SHARE_PCT AS EU_MARKET_SHARE_PCT WITH SYNONYMS = ('market share', 'share') COMMENT = 'Quarterly EU market share percentage',
    PRODUCT_METRICS.CLINICAL_SATISFACTION_SCORE AS CLINICAL_SATISFACTION_SCORE WITH SYNONYMS = ('satisfaction', 'clinical score') COMMENT = 'Clinical satisfaction score',
    PRODUCT_METRICS.NEW_SKU_LAUNCHES AS NEW_SKU_LAUNCHES WITH SYNONYMS = ('launches', 'new SKUs', 'new products') COMMENT = 'New SKU variants launched'
)

DIMENSIONS (
    PRODUCTS.PRODUCT_ID AS PRODUCT_ID COMMENT = 'Unique product identifier',
    PRODUCTS.PRODUCT_NAME AS PRODUCT_NAME WITH SYNONYMS = ('product', 'name', 'brand') COMMENT = 'Name of the MedTech product',
    PRODUCTS.MANUFACTURER AS MANUFACTURER WITH SYNONYMS = ('company', 'maker', 'supplier') COMMENT = 'Product manufacturer',
    PRODUCTS.HEADQUARTERS AS HEADQUARTERS WITH SYNONYMS = ('country', 'HQ') COMMENT = 'Manufacturer headquarters country',
    PRODUCTS.BUSINESS_AREA AS BUSINESS_AREA WITH SYNONYMS = ('division', 'business unit') COMMENT = 'Business area (Wound Care, OR Solutions, Gloves, Antiseptics)',
    PRODUCTS.PRODUCT_TYPE AS PRODUCT_TYPE WITH SYNONYMS = ('type', 'product type') COMMENT = 'Product type/format',
    PRODUCTS.PRICE_TIER AS PRICE_TIER WITH SYNONYMS = ('pricing', 'tier') COMMENT = 'Price category',
    PRODUCTS.IS_MOLNLYCKE_PRODUCT AS IS_MOLNLYCKE_PRODUCT WITH SYNONYMS = ('our product', 'molnlycke', 'ours', 'our products', 'our') COMMENT = 'TRUE = Mölnlycke product, FALSE = competitor',
    PRODUCTS.PRIMARY_CATEGORY AS PRIMARY_CATEGORY COMMENT = 'Primary product category',
    
    REVIEWS.REVIEW_ID AS REVIEW_ID COMMENT = 'Review identifier',
    REVIEWS.PRODUCT_CATEGORY AS PRODUCT_CATEGORY WITH SYNONYMS = ('category', 'wound care category') COMMENT = 'Product category reviewed',
    REVIEWS.REVIEWER_NAME AS REVIEWER_NAME COMMENT = 'Reviewer name',
    REVIEWS.REVIEWER_COUNTRY AS REVIEWER_COUNTRY WITH SYNONYMS = ('country', 'market', 'location') COMMENT = 'Reviewer country in Europe',
    REVIEWS.REVIEWER_ROLE AS REVIEWER_ROLE WITH SYNONYMS = ('role', 'clinician type', 'profession') COMMENT = 'Clinical role of the reviewer',
    REVIEWS.REVIEW_TITLE AS REVIEW_TITLE COMMENT = 'Review title',
    REVIEWS.REVIEW_TEXT AS REVIEW_TEXT WITH SYNONYMS = ('feedback', 'comment', 'review') COMMENT = 'Full review content',
    REVIEWS.CARE_SETTING AS CARE_SETTING WITH SYNONYMS = ('setting', 'care environment') COMMENT = 'Hospital Acute, Hospital Chronic, Community/Home Care, OR/Surgical',
    REVIEWS.WOUND_TYPE AS WOUND_TYPE WITH SYNONYMS = ('wound', 'indication') COMMENT = 'Type of wound treated',
    REVIEWS.REVIEW_DATE AS REVIEW_DATE COMMENT = 'Date posted',
    
    PRODUCT_METRICS.YEAR_QUARTER AS YEAR_QUARTER WITH SYNONYMS = ('quarter', 'period', 'date') COMMENT = 'Quarter'
)

METRICS (
    REVIEWS.AVG_RATING AS AVG(REVIEWS.RATING) COMMENT = 'Average rating',
    REVIEWS.REVIEW_COUNT AS COUNT(REVIEWS.REVIEW_ID) COMMENT = 'Number of reviews',
    PRODUCT_METRICS.TOTAL_UNITS_SOLD AS SUM(PRODUCT_METRICS.EU_UNITS_SOLD) COMMENT = 'Total units sold',
    PRODUCT_METRICS.TOTAL_REVENUE AS SUM(PRODUCT_METRICS.EU_REVENUE_EUR) COMMENT = 'Total revenue in EUR',
    PRODUCT_METRICS.AVG_MARKET_SHARE AS AVG(PRODUCT_METRICS.EU_MARKET_SHARE_PCT) COMMENT = 'Average market share'
)

COMMENT = 'Competitive intelligence for Mölnlycke Health Care in the European MedTech market'

AI_SQL_GENERATION 'Our products (Mölnlycke) have IS_MOLNLYCKE_PRODUCT = TRUE (10 products: Mepilex, Mepitel, Exufiber, Melgisorb, Avance Solo, Granudacyn, Biogel, BARRIER, ProcedurePak, Hibiclens). Competitors have IS_MOLNLYCKE_PRODUCT = FALSE (15 products). Always compare Mölnlycke product performance against competitors when relevant. Revenue and prices are in EUR. Key competitors: Smith+Nephew (ALLEVYN, PICO), ConvaTec (Aquacel), Coloplast (Biatain), Solventum (Tegaderm, V.A.C.).'

AI_QUESTION_CATEGORIZATION '
categorization_rules:
  - name: "employee_data"
    description: "Questions about employee salaries, personal information, or HR data"
    response: "I cannot provide information about employee data. Please contact HR directly."
  - name: "financial_internals"
    description: "Questions about internal profit margins, cost structures, or confidential financial details"
    response: "Detailed financial internals are confidential. I can help with general revenue and performance metrics."
  - name: "competitor_secrets"
    description: "Questions asking for confidential competitor strategies or non-public information"
    response: "I can only provide analysis based on publicly available data and our own operational metrics."
  - name: "patient_data"
    description: "Questions about specific patient outcomes, patient-identifiable information, or individual clinical cases"
    response: "I cannot provide patient-identifiable information. I can help with aggregated product performance data."
'
"""

print(f"Semantic view DDL ready ({len(semantic_view_ddl)} chars)")

In [ ]:
session.sql(semantic_view_ddl).collect()
print("Semantic view created: MOLNLYCKE_DEMO.ANALYTICS.MOLNLYCKE_COMPETITIVE_INTEL")

## Step 2: Test Cortex Analyst

In [ ]:
import requests
import json

def ask_analyst(question: str):
    token = session.connection._rest._token
    account = session.connection.account
    host = session.connection.host
    
    url = f"https://{host}/api/v2/cortex/analyst/message"
    
    headers = {
        "Authorization": f"Snowflake Token=\"{token}\"",
        "Content-Type": "application/json"
    }
    
    payload = {
        "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
        "semantic_view": "MOLNLYCKE_DEMO.ANALYTICS.MOLNLYCKE_COMPETITIVE_INTEL"
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    result = response.json()
    return result

def run_analyst_query(question: str):
    result = ask_analyst(question)
    
    sql = None
    for msg in result.get("message", {}).get("content", []):
        if msg.get("type") == "sql":
            sql = msg.get("statement")
            break
    
    if sql:
        print(f"Generated SQL:\n{sql}\n")
        return session.sql(sql).to_pandas()
    else:
        print("Response:", json.dumps(result, indent=2))
        return None

print("Cortex Analyst functions ready")

In [ ]:
result = run_analyst_query("How do our product ratings compare to competitors?")
display(result)

## Step 3: Create Cortex Search Service for Clinician Reviews

In [ ]:
session.sql("""
CREATE OR REPLACE CORTEX SEARCH SERVICE MOLNLYCKE_DEMO.ANALYTICS.PRODUCT_REVIEWS_SEARCH
ON REVIEW_TEXT
ATTRIBUTES PRODUCT_ID, PRODUCT_CATEGORY, CARE_SETTING
WAREHOUSE = AI_WH
TARGET_LAG = '1 day'
EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT 
        REVIEW_ID,
        PRODUCT_ID,
        PRODUCT_CATEGORY,
        REVIEWER_NAME,
        REVIEWER_COUNTRY,
        REVIEWER_ROLE,
        RATING,
        REVIEW_DATE,
        REVIEW_TITLE,
        REVIEW_TEXT,
        CARE_SETTING,
        WOUND_TYPE
    FROM MOLNLYCKE_DEMO.ANALYTICS.REVIEWS
)
""").collect()
print("Cortex Search Service created!")

In [ ]:
from snowflake.core import Root

molnlycke_products = session.sql("""
    SELECT PRODUCT_ID, PRODUCT_NAME FROM MOLNLYCKE_DEMO.ANALYTICS.DIM_PRODUCTS 
    WHERE IS_MOLNLYCKE_PRODUCT = TRUE
""").collect()
molnlycke_ids = [row['PRODUCT_ID'] for row in molnlycke_products]
print(f"Mölnlycke products: {[row['PRODUCT_NAME'] for row in molnlycke_products]}")
print(f"PRODUCT_IDs: {molnlycke_ids}\n")

root = Root(session)
search_service = (root
    .databases["MOLNLYCKE_DEMO"]
    .schemas["ANALYTICS"]
    .cortex_search_services["PRODUCT_REVIEWS_SEARCH"]
)

filter_conditions = {"@or": [{"@eq": {"PRODUCT_ID": pid}} for pid in molnlycke_ids]}

results = search_service.search(
    query="complaints about adhesion and conformability on foam dressings",
    columns=["REVIEW_TEXT", "RATING", "REVIEWER_NAME", "REVIEW_TITLE", "PRODUCT_ID", "PRODUCT_CATEGORY"],
    filter=filter_conditions,
    limit=5
)

print("Search: 'complaints about adhesion and conformability on foam dressings' (MÖLNLYCKE ONLY)\n")
for r in results.results:
    print(f"Rating: {r['RATING']} - {r['REVIEW_TITLE']} (Product ID: {r['PRODUCT_ID']}, {r['PRODUCT_CATEGORY']})")
    print(f"   {r['REVIEW_TEXT'][:200]}...")
    print()

## Step 4: Create Cortex Agent

In [ ]:
session.sql("""
CREATE OR REPLACE AGENT MOLNLYCKE_DEMO.ANALYTICS.MOLNLYCKE_PRODUCT_ANALYST
  COMMENT = 'Competitive intelligence analyst for Mölnlycke Health Care in European MedTech markets'
  PROFILE = '{
    "display_name": "Mölnlycke Product Analyst"
  }'
  FROM SPECIFICATION $$
  {
    "models": {
      "orchestration": "claude-sonnet-4-5"
    },
    "instructions": {
      "orchestration": "You help analyze MedTech product competitive intelligence for Mölnlycke Health Care across European markets. Use the analyst tool for data queries (ratings, metrics, market share, revenue comparisons). Use the search tool to find specific clinician reviews or product feedback. Our products have IS_MOLNLYCKE_PRODUCT=TRUE (Mepilex, Mepitel, Exufiber, Melgisorb, Avance Solo, Granudacyn, Biogel, BARRIER, ProcedurePak, Hibiclens). Competitors have IS_MOLNLYCKE_PRODUCT=FALSE. Key competitors: Smith+Nephew (ALLEVYN, PICO, ACTICOAT), ConvaTec (Aquacel, Aquacel Foam, Aquacel Ag+), Coloplast (Biatain, Biatain Ag), Solventum (Tegaderm, V.A.C.), B. Braun (Askina, Prontosan).",
      "response": "Be concise and data-driven. When discussing reviews, include relevant clinician quotes. Always specify product names and manufacturers rather than just IDs."
    },
    "tools": [
      {
        "tool_spec": {
          "type": "cortex_analyst_text_to_sql",
          "name": "product_analyst",
          "description": "Query MedTech competitive intelligence data including product ratings, market share, revenue, units sold, and aggregated statistics across European markets"
        }
      },
      {
        "tool_spec": {
          "type": "cortex_search",
          "name": "review_search",
          "description": "Search clinician product reviews to find specific clinical feedback, complaints, or praise about wound care and surgical products"
        }
      }
    ],
    "tool_resources": {
      "product_analyst": {
        "semantic_view": "MOLNLYCKE_DEMO.ANALYTICS.MOLNLYCKE_COMPETITIVE_INTEL",
        "execution_environment": {
          "type": "warehouse",
          "warehouse": "AI_WH"
        }
      },
      "review_search": {
        "search_service": "MOLNLYCKE_DEMO.ANALYTICS.PRODUCT_REVIEWS_SEARCH",
        "max_results": 10
      }
    }
  }
  $$
""").collect()
print("Cortex Agent created with Analyst + Search tools!")

In [ ]:
session.sql("GRANT USAGE ON AGENT MOLNLYCKE_DEMO.ANALYTICS.MOLNLYCKE_PRODUCT_ANALYST TO ROLE PUBLIC").collect()

try:
    session.sql("""
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT 
ADD AGENT MOLNLYCKE_DEMO.ANALYTICS.MOLNLYCKE_PRODUCT_ANALYST
""").collect()
    print("Agent added to Snowflake Intelligence!")
except Exception as e:
    if "already present" in str(e):
        print("Agent already registered in Snowflake Intelligence, skipping.")
    else:
        raise
print()
print("Access at: https://ai.snowflake.com")

## Step 5: Test Agent

In [ ]:
def ask_agent(question: str):
    row = session.sql(f"""
    SELECT TRY_PARSE_JSON(
        SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
            'MOLNLYCKE_DEMO.ANALYTICS.MOLNLYCKE_PRODUCT_ANALYST',
            $${{
                "messages": [
                    {{
                        "role": "user",
                        "content": [
                            {{"type": "text", "text": "{question}"}}
                        ]
                    }}
                ]
            }}$$
        )
    ) AS resp
    """).collect()[0]['RESP']
    
    if isinstance(row, str):
        result = json.loads(row)
    else:
        result = row
    
    if result and isinstance(result, dict) and 'content' in result:
        for item in result['content']:
            if isinstance(item, dict) and item.get('type') == 'text':
                return {"response": item.get('text', ''), "raw": result}
    
    return {"response": "", "raw": result}

print("Function defined - run next cell to test")

In [ ]:
result = ask_agent("How do our product ratings compare to Smith+Nephew and ConvaTec?")
print("Response:", result.get('response', '')[:500])
print("\nFull result:")
print(json.dumps(result.get('raw'), indent=2, default=str))

In [ ]:
test_questions = [
    # Zlatko (CEO)
    "How do our product ratings compare to Smith+Nephew and ConvaTec?",
    "Which of our products is underperforming in the European market?",
    "What would it take to get Mepilex to a 4.5 average rating?",
    # Anders (EVP Wound Care)
    "Which wound care category has the biggest gap vs competitors?",
    "What do clinicians praise about Coloplast Biatain that they don't mention about Mepilex?",
    # Maria (CPO/Brand)
    "How is our brand perceived vs Smith+Nephew in the UK market?",
    # Eric (COO)
    "What are the top product quality complaints across our foam dressings?",
    "Which product features should we prioritise in R&D based on clinician feedback?",
    "How do Mepilex Border reviews compare to ConvaTec Aquacel Foam?",
    # Agent Testing
    "How is Mepilex performing compared to ALLEVYN?",
    "What should we improve on Avance Solo NPWT?",
    "Where should we invest - foam dressings or NPWT?"]